In [ ]:
from pelmesha.pspectra import DataProc_1d, peaks_prop_infunc,msalign,smoothing, Configs, get_mz_discretion_coeffs
from pelmesha.loaders import find_paths
from pelmesha.pfeats import Pgrouping_KD, Getrefpeaks, Pgrouping_KD_table, mspeaks_KD
import pandas as pd
import xarray as xr
import numpy as np
from multiprocessing import cpu_count
from itertools import pairwise, product
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from joblib import Parallel, delayed
import math
import os
class AbsoluteFormatter(ticker.ScalarFormatter):
    def _set_format(self):
        # Этот метод вызывается внутренне для настройки формата
        super()._set_format()
        
    def __call__(self, x, pos=None):
        # Главный метод: берем abs(x) и отдаем стандартному родителю
        return super().__call__(abs(x), pos)


def ESI_peaks(path, configs, signal_distortions, scan_filter = None, free_cores = 2, draw = True, add_zero_points = True, grouping_peaks = True, **Pgrouping_KD_configs): #TODO: Не универсальная функция!!!! Пока только локальная
    """
    path: path to netcdf file
    configs: configs for DataProc_1d and peaks_prop_infunc
    signal_distortions: list of tuples with signal distortions
    scan_filter: list of scan filters
    free_cores: number of free cores
    draw: draw plots
    Pgrouping_KD_configs: configs for Pgrouping_KD
    """

    ds = xr.open_dataset(path, engine='netcdf4')
    mz = ds.variables['mass_values'].values
    intens = ds.variables['intensity_values'].values
    file_name = os.path.basename(path).split('.')[0]
    scan_dots_sequence = np.append(ds['scan_index'].values, len(mz))
    scan_dots_slice = [slice(*x,1) for x in pairwise(scan_dots_sequence)]
    scan_sequence = np.array(range(len(ds['scan_filters'])))

    DataProc_configs = configs["DataProc_configs"]
    PeakPicking_configs = configs["peaks_configs"]
    peaklists = {}

    mz_min = min(mz)
    mz_max = max(mz)
    if scan_filter is None:
        scan_filter = np.unique(ds['scan_filters'])
    elif isinstance(scan_filter, (int, np.int32, np.int64)):
        scan_filter = [scan_filter]

    filtered_scans = [[i, scan_sequence[np.array(ds['scan_filters'].values == i, dtype=bool)]] for i in np.unique(ds['scan_filters']) if i in scan_filter]
    
    for i, scans in filtered_scans:
        # data = [0]*len(scans)
        mzs = []
        ints = []
        min_discret = np.inf
        for n, scan in enumerate(scans):
            slc = scan_dots_slice[scan] 
            loc_mz = mz[slc]
            loc_ints = intens[slc]
            mzs.append(loc_mz.copy())
            ints.append(loc_ints.copy())
            min_discret = min(min_discret, np.diff(loc_mz).min())
        discret_coeffs = get_mz_discretion_coeffs(np.sort(np.unique(np.hstack(mzs))), min_discret, draw = False)
        # gen_spectrum = pd.DataFrame(np.vstack([np.concatenate(mzs), np.concatenate(ints)]).T, columns=['mz','int']).groupby('mz').agg(['mean', 'median']).reset_index()
        # gen_mz, gen_int_mean = add_zero_points_to_peaks(gen_spectrum['mz'].values, gen_spectrum['int']['mean'].values, discret_coeffs)
        # gen_mz, gen_int_median = add_zero_points_to_peaks(gen_spectrum['mz'].values, gen_spectrum['int']['median'].values, discret_coeffs)
        # gen_spectrum = pd.DataFrame(np.vstack([gen_mz,gen_int_mean, gen_int_median]).T, columns=pd.MultiIndex.from_tuples([('mz',''), ('int', 'mean'), ('int', 'median')]))

        gen_spectrum = pd.DataFrame(np.vstack([np.concatenate(mzs), np.concatenate(ints)]).T, columns=['mz','int']).groupby('mz').median().reset_index()
        gen_mz, gen_int_median = add_zero_points_to_peaks(gen_spectrum['mz'].values, gen_spectrum['int'].values, discret_coeffs)
        gen_spectrum = pd.DataFrame({'mz': gen_mz, "int" : gen_int_median})
        if add_zero_points or configs['resample_to_dots'] or (i in [2,3]):
            for n, (loc_mz, loc_ints) in enumerate(zip(mzs, ints)):
                if add_zero_points:
                    loc_mz, loc_ints = add_zero_points_to_peaks(loc_mz, loc_ints, discret_coeffs)
                if configs['resample_to_dots']:
                    resampled_mz = np.linspace(mz_min, mz_max,configs['resample_to_dots'])
                    loc_ints = np.interp(resampled_mz, loc_mz, loc_ints)
                    loc_mz = resampled_mz
                if i in [2,3]:
                    artefacts_bool = np.zeros(len(loc_mz), dtype=bool)
                    for distortion in signal_distortions:
                        artefacts_bool = artefacts_bool | ((loc_mz > distortion[0]) & (loc_mz < distortion[1]))
                        loc_ints[artefacts_bool] = 0
                mzs[n], ints[n] = loc_mz, loc_ints
        
        data = list(zip(mzs, ints, scans))

        mzs = np.sort(np.unique(np.hstack(mzs)))
        discret_coeffs = get_mz_discretion_coeffs(mzs, min_discret)
        par_args = list(product(data,[DataProc_configs],[PeakPicking_configs]))
        
        peaklists[i] = Parallel(n_jobs= cpu_count()-free_cores)(delayed(ESI_int2proc2peaklist_parbatched)(*spectrum) for spectrum in par_args)
        peaklists[i] = pd.DataFrame(np.vstack(peaklists[i]), columns = PeakPicking_configs['headers'])
        peaklists[i].attrs.update({'headers': PeakPicking_configs['headers'], 'discret_coeffs': discret_coeffs})
        if grouping_peaks:
            peaklists[i] = Pgrouping_KD(peaklists[i], sample = f'{file_name}', roi = f"Scan filter: {i}", draw = draw,**Pgrouping_KD_configs)
            if draw:
                formatter = AbsoluteFormatter(useMathText=True)
                formatter.set_scientific(True)
                formatter.set_powerlimits((-3, 3)) # При каких степенях выносить множитель

                plt.figure(num=plt.get_fignums()[-1])
                ax2pr = plt.gca()
                ylim_min, ylim_max = ax2pr.get_ylim()
                ax2pr.set_ylim((-max((abs(ylim_min), ylim_max)), max((abs(ylim_min), ylim_max))) )
                
                ax2 = ax2pr.twinx()
                ax2.yaxis.set_major_formatter(formatter)

                mz_draw_borders = plt.xlim()
                gen_mz_bool = (gen_spectrum['mz'] > mz_draw_borders[0]) & (mz_draw_borders[1] > gen_spectrum['mz'])
                plt.plot(gen_spectrum.loc[gen_mz_bool, 'mz'],gen_spectrum.loc[gen_mz_bool,'int']*(-1), color = 'g', alpha = 0.75)
                rand_spec_1 = Pgrouping_KD_table.rand_spec_1
                rand_spec_2 = Pgrouping_KD_table.rand_spec_2
                if rand_spec_1:
                        
                    rand_spec = rand_spec_1[-1]
                    if not isinstance(rand_spec, int):
                        rand_spec = int(rand_spec)
                    
                    plt.figure(num=plt.get_fignums()[-2])
                    ax1pr = plt.gca() 
                    ylim_min, ylim_max = ax1pr.get_ylim() 
                    ax1pr.set_ylim((-max((abs(ylim_min), ylim_max)), max((abs(ylim_min), ylim_max))) )
                    ax1 = ax1pr.twinx()
                    ax1.yaxis.set_major_formatter(formatter)
                    mz_draw_borders = plt.xlim()
                    gen_mz_bool = (gen_spectrum['mz'] > mz_draw_borders[0]) & (mz_draw_borders[1] > gen_spectrum['mz'])
                    plt.plot(gen_spectrum.loc[gen_mz_bool, 'mz'],gen_spectrum.loc[gen_mz_bool,'int']*(-1), color = 'g', alpha = 0.75)
                if isinstance(draw,(list,tuple)):
                    iter_draw = draw[1]
                else:
                    iter_draw = 1
                # Leg_1 = ['Mean spectrum', 'Median spectrum']
                # Leg_2 = ['Mean spectrum', 'Median spectrum']
                Leg_1 = ['Median spectrum']
                Leg_2 = ['Median spectrum']
                
                
                for it in range(iter_draw):
                    if Pgrouping_KD_table.rand_spec_1:
                        rand_spec_1 = [np.random.choice(next(scans for filter, scans in filtered_scans if filter == i))]
                    else:
                        rand_spec_1 = None
                    rand_spec_2 = np.random.choice(next(scans for filter, scans in filtered_scans if filter == i))
                    if rand_spec_1:
                        
                        rand_spec = rand_spec_1[-1]
                        if not isinstance(rand_spec, int):
                            rand_spec = int(rand_spec)
                        
                        x,y,scan = next([x,y,scan] for x,y,scan in data if scan == rand_spec)                    

                        dots_distance = np.quantile(np.diff(x),0.33)

                        DataProc_configs['smoothing_configs']['smooth_window'](dots_distance) 
                        DataProc_configs['msalign_configs']['shift_range'](dots_distance)
                        DataProc_configs['baseliner'](x)
                        mz_draw_borders = plt.xlim()
                        dots_bord_spec = (x>=mz_draw_borders[0]) & (x<=mz_draw_borders[1])
                        plt.figure(num=plt.get_fignums()[-2])
                        ax1.plot(x[dots_bord_spec],y[dots_bord_spec],alpha=0.5)
                        proc_y = DataProc_1d(y, x,**DataProc_configs)
                        dots_bord_spec = (x>=mz_draw_borders[0]) & (x<=mz_draw_borders[1])
                        ax1.plot(x[dots_bord_spec], proc_y[dots_bord_spec],alpha=0.5)
                        plt.gcf().tight_layout()
                        ax1.set_ylabel("Intensity")
                        Leg_1+=[f"Raw mass spectrum N {rand_spec}", f"Processed mass spectrum N {rand_spec}"]
                        
                        plt.legend(Leg_1,loc='upper left')

                        ylim_min, ylim_max = ax1.get_ylim() 
                        ax1.set_ylim((-max((abs(ylim_min), ylim_max)), max((abs(ylim_min), ylim_max))) ) 
                    if not isinstance(rand_spec_2, int):
                        rand_spec_2 = int(rand_spec_2)

                    x,y,scan = next([x,y,scan] for x,y,scan in data if scan == rand_spec_2)
                    dots_distance = np.quantile(np.diff(x),0.33)
                    plt.figure(num=plt.get_fignums()[-1])
                    DataProc_configs['smoothing_configs']['smooth_window'](dots_distance) 
                    DataProc_configs['msalign_configs']['shift_range'](dots_distance)
                    DataProc_configs['baseliner'](x)
                    mz_draw_borders = plt.xlim()
                    dots_bord_spec = (x>=mz_draw_borders[0]) & (x<=mz_draw_borders[1])
                    ax2.plot(x[dots_bord_spec],y[dots_bord_spec],alpha=0.5)
                    proc_y = DataProc_1d(y, x,**DataProc_configs)
                    dots_bord_spec = (x>=mz_draw_borders[0]) & (x<=mz_draw_borders[1])
                    ax2.plot(x[dots_bord_spec], proc_y[dots_bord_spec],alpha=0.5)

                    plt.gcf().tight_layout()
                    ax2.set_ylabel("Intensity")
                    Leg_2 += [f"Raw mass spectrum N {rand_spec_2}", f"Processed mass spectrum N {rand_spec_2}"]
                    ax2.legend(Leg_2, loc='upper left')
                    rand_spec_1 = False
                
                ylim_min, ylim_max = ax2.get_ylim() 
                ax2.set_ylim((-max((abs(ylim_min), ylim_max)), max((abs(ylim_min), ylim_max))) )
                plt.show()

    return peaklists
            

def ESI_int2proc2peaklist_parbatched(data, DataProc_configs, PeakPicking_configs):
    """
    Process a mass spectrum and return a peaklist.

    Parameters
    ----------
    data : tuple
        Tuple of (mz, ints, scan)
    DataProc_configs : dict
        Dictionary of configuration parameters for data processing
    PeakPicking_configs : dict
        Dictionary of configuration parameters for peak picking

    Returns
    -------
    peaklist : list
        List of tuples (mz, ints, scan, pk_x, left, right)
    """
    mz, ints, scan = data
    dots_distance = np.quantile(np.diff(mz),0.33)

    DataProc_configs['smoothing_configs']['smooth_window'](dots_distance) 
    DataProc_configs['msalign_configs']['shift_range'](dots_distance)
    DataProc_configs['baseliner'](mz)
    
    proc_ints = DataProc_1d(ints, mz,**DataProc_configs)
    peaklist = peaks_prop_infunc(mz, proc_ints, np.where(np.diff(proc_ints) != 0)[0], len(mz),
                                scan, **PeakPicking_configs)

    return peaklist
def peaks_prop(X, Y,oversegmentationfilter=None,peaklocation=1, return_pkY = False):
    """
    Detect peaks in a KDE curve and return their centers and boundaries.

    Parameters
    ----------
    X : ndarray
        Monotonic array of X coordinates (e.g., m/z grid).
    Y : ndarray
        Corresponding density/height values.
    oversegmentation_filter : float or None, optional
        Minimal allowed separation between adjacent peaks; when provided, peaks
        closer than this threshold are merged.
    peak_location : float, optional
        Fraction of the peak height to compute a barycentric center; used in
        boundary calculations as a threshold. Default is 1.

    Returns
    -------
    pk_x : ndarray
        Estimated peak centers (X positions). May contain NaNs if a region has
        no samples above the threshold.
    left : ndarray
        Left boundary (valley position) for each peak.
    right : ndarray
        Right boundary (valley position) for each peak.
    """
    n = X.size
    # Robust valley finding
    valley_dots = np.concatenate((np.where(np.diff(Y) != 0)[0], [n-1]))    
    loc_min = np.diff(Y[valley_dots])
    loc_min = (np.array([True,*(loc_min < 0)])) & np.array(([*(loc_min > 0),True]))
    left_min = np.concatenate([[-1],valley_dots[:-1]])[loc_min][:-1] + 1
    right_min = valley_dots[loc_min][1:]
    # Compute max and min for every peak
    size = left_min.shape
    val_max = np.empty(size)
    pos_peak = np.empty(size)
    for idx, [lm, rm] in enumerate(zip(left_min, right_min)): 
        val_max[idx] = np.max(Y[lm:rm]) 
        pos_peak[idx] = lm + np.argmax(Y[lm:rm])
    
    # Remove oversegmented peaks
    if oversegmentationfilter:
        while True:
            peak_thld = val_max * peaklocation - math.sqrt(np.finfo(float).eps)
            pkX = np.empty(left_min.shape)
            
            for idx, [lm, rm, th] in enumerate(zip(left_min, right_min, peak_thld)):
                mask = Y[lm:rm] >= th
                if not mask.any():
                    pkX[idx]=np.nan
                else:
                    pkX[idx] = np.sum(Y[lm:rm][mask] * X[lm:rm][mask]) / np.sum(Y[lm:rm][mask])
            dpkX = np.concatenate(([np.inf], np.diff(pkX), [np.inf]))
            
            j = np.where((dpkX[1:-1] <= oversegmentationfilter) & (dpkX[1:-1] <= dpkX[:-2]) & (dpkX[1:-1] < dpkX[2:]))[0]
            if j.size == 0:
                break
            left_min = np.delete(left_min, j + 1)
            right_min = np.delete(right_min, j)
            stack_j = np.vstack((j,j+1))
            pos_peak_oversegmentation = pos_peak[stack_j]
            val_max_oversegmentation = val_max[stack_j]
            max_idx = np.argmax(val_max_oversegmentation, axis = 0)
            range_j = np.arange(len(j))
            val_max[j] = val_max_oversegmentation[max_idx, range_j]
            pos_peak[j] = pos_peak_oversegmentation[max_idx, range_j]
            val_max = np.delete(val_max, j + 1)
            pos_peak = np.delete(pos_peak, j + 1)
    else:
        peak_thld = val_max * peaklocation - math.sqrt(np.finfo(float).eps)
        pkX = np.empty(left_min.shape)
        
        for idx, [lm, rm, th] in enumerate(zip(left_min, right_min, peak_thld)):
            mask = Y[lm:rm] >= th
            if not mask.any():
                pkX[idx]=np.nan
            else:
                pkX[idx] = np.sum(Y[lm:rm][mask] * X[lm:rm][mask]) / np.sum(Y[lm:rm][mask])
    if return_pkY:
        return np.array((pkX, X[left_min], X[right_min], val_max)) 
    return np.array((pkX, X[left_min], X[right_min]))


def add_zero_points_to_peaks(mz, ints, mz_discret_coeffs):
    """
    Add zero points to peaks
    """
    mz_discretion_model = np.poly1d(mz_discret_coeffs)

    diff_mz = np.diff(mz)
    mz_discr = mz_discretion_model(mz[:-1])
    big_gap_bool = diff_mz > 3.5*mz_discr
    small_gap_bool = (diff_mz > 1.75*mz_discr) ^ big_gap_bool
    new_val = {}
    if np.any(big_gap_bool):
        new_val['left'] = mz[np.append(big_gap_bool, [False])] + mz_discr[big_gap_bool]
        new_val['right'] = mz[np.append([False], big_gap_bool)] - mz_discr[big_gap_bool]

    if np.any(small_gap_bool):
        new_val['small'] = mz[np.append(small_gap_bool, [False])] + mz_discr[small_gap_bool]

    if new_val:
        new_val['borders'] = [mz[0]-mz_discr[0], mz[-1]+mz_discr[-1]]
        new_val = np.concatenate(list(new_val.values()), axis=None)
        idx = np.searchsorted(mz, new_val)
        mz = np.insert(mz, idx, new_val)
        ints = np.insert(ints, idx, 0)
        
    # sorting by mz
    idx_sort = np.argsort(mz)
    new_loc_mz = mz[idx_sort]
    new_loc_ints = ints[idx_sort]

    return new_loc_mz, new_loc_ints

In [ ]:
# Наблюдаемые диапазоны наводок орбитрепа (список неполный и недостаточно расширенный). Интенсивности в этих диапазонах зануляются в процессе обработки.
signal_distortions = [(99, 100.2),
                      (102.09, 103.86),
                      (107.79, 109.4),
                      (112.65, 112.9),
                      (123.2, 123.6),
                      (127, 128.2),
                      (135.4, 136.12), 
                     (155.7, 156.6), 
                     (168.5, 170.8),
                     (188.75, 189.15),
                     (202.00, 202.55),
                     (203.13, 204.65),
                     (206.4, 206.8), 
                     (210.30, 210.60), 
                     (220.10, 221.02), 
                     (229.36, 231.53), 
                     (241.92, 243.0),
                     (277.4, 278.4), 
                     (285.83, 287.11),
                     (321.6, 322.0),
                     (375.0, 375.6),
                     (398.18, 401.15), 
                     (472.29, 472.89), 
                     (508.34, 512.61), 
                     (620.00, 622.81), 
                     (643.60, 645.03), 
                     (674.0, 676.17), 
                     (677.37, 683.2), 
                     (906.92, 908.83), 
                     (510.0, 511.2), 
                     (622.5, 627.5), 
                     (643.0, 644.5), 
                     (676.0, 679.5)]
path = r'C:\Job_and_Literature\24.cdf'


In [ ]:
# from pelmesha.pspectra import DataProc_1d, peaks_prop_infunc,msalign,smoothing, Configs, get_mz_discretion_coeffs
# from pelmesha.pfeats import Pgrouping_KD, Getrefpeaks, Pgrouping_KD_table, mspeaks_KD
# import pandas as pd
# import xarray as xr
# import numpy as np
# from multiprocessing import cpu_count
# from itertools import pairwise, product
# import matplotlib.pyplot as plt
# from joblib import Parallel, delayed
# from pelmesha.pspectra import peaks_prop_array
# import math
# import matplotlib.ticker as ticker

# scan_filter = None
# ds = xr.open_dataset(path, engine='netcdf4')
# mz = ds.variables['mass_values'].values
# intens = ds.variables['intensity_values'].values

# scan_dots_sequence = np.append(ds['scan_index'].values, len(mz))
# scan_dots_slice = [slice(*x,1) for x in pairwise(scan_dots_sequence)]
# scan_sequence = np.array(range(len(ds['scan_filters'])))

# peaklists = {}

# mz_min = min(mz)
# mz_max = max(mz)
# if scan_filter is None:
#     scan_filter = np.unique(ds['scan_filters'])
# elif isinstance(scan_filter, (int, np.int32, np.int64)):
#     scan_filter = [scan_filter]

# filtered_scans = [[i, scan_sequence[np.array(ds['scan_filters'].values == i, dtype=bool)]] for i in np.unique(ds['scan_filters']) if i in scan_filter]

# for i, scans in filtered_scans:
#     # data = [0]*len(scans)
#     mzs = []
#     ints = []
#     min_discret = np.inf
#     for n, scan in enumerate(scans):
#         slc = scan_dots_slice[scan] 
#         loc_mz = mz[slc]
#         loc_ints = intens[slc]
#         mzs.append(loc_mz.copy())
#         ints.append(loc_ints.copy())
#         min_discret = min(min_discret, np.diff(loc_mz).min())
#     discret_coeffs = get_mz_discretion_coeffs(np.sort(np.unique(np.hstack(mzs))), min_discret, draw = False)
#     for n, (loc_mz, loc_ints) in enumerate(zip(mzs, ints)):
#         loc_mz, loc_ints = add_zero_points_to_peaks(loc_mz, loc_ints, discret_coeffs)
#         if i in [2,3]:
#             artefacts_bool = np.zeros(len(loc_mz), dtype=bool)
#             for distortion in signal_distortions:
#                 artefacts_bool = artefacts_bool | ((loc_mz > distortion[0]) & (loc_mz < distortion[1]))
#                 loc_ints[artefacts_bool] = 0
#         mzs[n], ints[n] = loc_mz, loc_ints
#     gen_spectrum = pd.DataFrame(np.vstack([np.concatenate(mzs), np.concatenate(ints)]).T, columns=['mz','int']).groupby('mz').agg(['mean', 'median']).reset_index()
#     plt.figure(figsize=(25,4))
#     class AbsoluteFormatter(ticker.ScalarFormatter):
#         def _set_format(self):
#             # Этот метод вызывается внутренне для настройки формата
#             super()._set_format()
            
#         def __call__(self, x, pos=None):
#             # Главный метод: берем abs(x) и отдаем стандартному родителю
#             return super().__call__(abs(x), pos)
#     gen_mz_bool = 
#     plt.plot(gen_spectrum['mz'],gen_spectrum['int']['mean']*(-1), color = 'r', alpha = 0.75)
#     plt.plot(gen_spectrum['mz'],gen_spectrum['int']['median']*(-1), color = 'g', alpha = 0.75)
    
#     ax = plt.gca()
#     formatter = AbsoluteFormatter(useMathText=True)
#     formatter.set_scientific(True)
#     formatter.set_powerlimits((-3, 3)) # При каких степенях выносить множитель
#     ax.yaxis.set_major_formatter(formatter)
#     # Создаем нормальный класс, который просто меняет знак перед отрисовкой



In [ ]:
# def split_spectrum_test(loc_mz, loc_ints, scale = 5):
#     """
#     Split spectrum
#     """
#     if len(loc_mz) == 1:
#         return [(loc_mz, loc_ints)]
#     min_distance2split = np.median(np.diff(loc_mz))
#     nums_distance_bool = np.diff(loc_mz) > scale*min_distance2split
#     left_nums = loc_mz[:-1][nums_distance_bool]
#     right_nums = loc_mz[1:][nums_distance_bool] 
#     mz_bins = np.concatenate(([loc_mz[0] - min_distance2split], (left_nums + right_nums)/2, [loc_mz[-1] + min_distance2split]))
#     batched_spectrum = [0]*(len(mz_bins) - 1)
#     for n_batch, mz_bin in enumerate(pairwise(mz_bins)):
#         mask = (loc_mz > mz_bin[0]) & (loc_mz < mz_bin[1])
#         batched_spectrum[n_batch] = (loc_mz[mask], loc_ints[mask])
#     return batched_spectrum

# def add_zero_points_to_peaks_test(mz, ints):
#     """
#     Add zero points to peaks
#     """
#     splitted_spectrum = split_spectrum(mz, ints, scale= 10)
#     splitted_spectrum = [result for spec in splitted_spectrum for result in split_spectrum(*spec, scale= 5)]

#     new_loc_mz = np.array([])
#     new_loc_ints = np.array([])
#     for loc_mz, loc_ints in splitted_spectrum:
#         data = peaks_prop(loc_mz, loc_ints,return_pkY=True, oversegmentationfilter=1)

#         for n in range(len(data[0,:])):
#             peak_base = (data[1,n], data[2,n])
#             peak_bool = (loc_mz >= peak_base[0]) & (loc_mz <= peak_base[1])
#             peaks_dots = loc_mz[peak_bool]
#             median_dist = np.quantile(np.diff(peaks_dots),0.25)
#             # add dot to left
#             bool_2xdist_left =  loc_mz >= (peak_base[0] - 4*median_dist)
#             bool_2right_left = loc_mz < peak_base[0]  
#             if (bool_2xdist_left & bool_2right_left).sum() < 2:
#                 new_val = peak_base[0] - 2*median_dist
#                 idx = np.searchsorted(loc_mz, new_val)
#                 loc_mz = np.insert(loc_mz, idx, new_val)
#                 loc_ints = np.insert(loc_ints, idx, 0)
#             # add dot to right
#             bool_2xdist_right =  loc_mz <= (peak_base[1] + 4*median_dist)
#             bool_2left_right = loc_mz > peak_base[1] 
#             if (bool_2xdist_right & bool_2left_right).sum() < 2:
#                 new_val = peak_base[1] + 2*median_dist
#                 idx = np.searchsorted(loc_mz, new_val)
#                 loc_mz = np.insert(loc_mz, idx, new_val)
#                 loc_ints = np.insert(loc_ints, idx, 0)
#         new_loc_mz = np.concatenate((new_loc_mz, loc_mz), axis=None)
#         new_loc_ints = np.concatenate((new_loc_ints, loc_ints), axis=None)
#     idx_sort = np.argsort(new_loc_mz)
#     new_loc_mz = new_loc_mz[idx_sort]
#     new_loc_ints = new_loc_ints[idx_sort]
    
#     data = peaks_prop(new_loc_mz, new_loc_ints,return_pkY=True)
#     max_y = max(data[3,:])
#     plt.figure(figsize=(25,5))
#     plt.scatter(data[0,:], data[3,:])
#     plt.vlines(data[1,:], ymin=0, ymax=max_y,colors='r')
#     plt.vlines(data[2,:], ymin=0, ymax=max_y, colors='g')
#     xlim = plt.xlim(277.5, 278.75)
#     bool_segment = (np.array(new_loc_mz) > xlim[0]) & (np.array(new_loc_mz) < xlim[1])
#     segment_y = new_loc_ints[bool_segment] 
#     plt.ylim(0, segment_y.max())
#     plt.plot(new_loc_mz, new_loc_ints,marker='x',color='k')
#     # plt.vlines(new_loc_mz[data[4,:].astype(int)], ymin=0, ymax=max_y, colors='b')
#     plt.show()
#     return new_loc_mz, new_loc_ints
# # add_zero_points_to_peaks(*add_zero_points_to_peaks(loc_mz, loc_ints))
# display(np.array(loc_mz), np.array(loc_ints))
# add_zero_points_to_peaks_test(loc_mz, loc_ints)
    

# # Попробовать проделать тоже самое с разбитым спектром по медианной дистанции

# Фильтрация шумовых пиков Orbitrap
На основе [данной работы](https://pubs.acs.org/doi/10.1021/ac403278t)

In [ ]:
# Создаём итоговый пик-лист после выравнивания и без выравнивания (просто для сравнения)
peaklists = {}
account_mzscale = True
path_folder = r"C:\Job_and_Literature\ESI_test"
for path in find_paths(path_folder, file_end = '.cdf'):
    for i in [2,3]:
        if i == 2 or i == 3: # Настройки для обработки Орбитреп данных делаем без какой-либо фильтрации пиков, фильтр шумовых пиков сделаем далее. Отстутсвие фильтра на этом этапе - ключевой момент, так как дальнейший фильтр основывается на статистике
            oversegmentationfilter = 0
            SNR_threshold = 0
            resample_to_dots = None #150000
            CountF = 0

        configs = Configs([DataProc_1d,peaks_prop_infunc,msalign,smoothing],
                        align_peaks = None,
                        align_pweights = None,
                        resample_to_dots = resample_to_dots,
                    #   smooth_algo = 'GA', 
                    #   smooth_window = 0.5, 
                    #   baseline_algo = 'asls
                    SNR_threshold = SNR_threshold, 
                    oversegmentationfilter = oversegmentationfilter)
        peaklists.update(ESI_peaks(path, configs,signal_distortions,i, draw_borders = 1.5, dupl_drop = False, account_mzscale = account_mzscale, CountF = CountF, return_pkY = True, draw = (True,2)))

In [ ]:
# Создаём итоговый пик-лист после выравнивания и без выравнивания (просто для сравнения)
peaklists = {}
account_mzscale = True
for i in [2,3]:
    if i == 2 or i == 3: # Настройки для обработки Орбитреп данных делаем без какой-либо фильтрации пиков, фильтр шумовых пиков сделаем далее. Отстутсвие фильтра на этом этапе - ключевой момент, так как дальнейший фильтр основывается на статистике
        oversegmentationfilter = 0
        SNR_threshold = 0
        resample_to_dots = None #150000
        CountF = 0

    configs = Configs([DataProc_1d,peaks_prop_infunc,msalign,smoothing],
                      align_peaks = None,
                      align_pweights = None,
                      resample_to_dots = resample_to_dots,
                #   smooth_algo = 'GA', 
                #   smooth_window = 0.5, 
                #   baseline_algo = 'asls
                  SNR_threshold = SNR_threshold, 
                  oversegmentationfilter = oversegmentationfilter)
    peaklists.update(ESI_peaks(path, configs,signal_distortions,i, draw_borders = 1.5, dupl_drop = False, account_mzscale = account_mzscale, CountF = CountF, return_pkY = True, draw = (True,2)))

In [ ]:
for i in peaklists:
    peaklists[i]['FWHM'] = peaklists[i]['FWHMR'] - peaklists[i]['FWHML']

In [ ]:
for i in peaklists:
    peaklists[i].plot(x='Peak', y = 'FWHM',figsize=(25, 6))
    peaklists[i].plot(x='mz', y = 'FWHM',figsize=(25, 6))
    peaklists[i].plot(x='mz', y = 'KD_bandwidth',figsize=(25, 6))


In [ ]:
from KDEpy import FFTKDE
from scipy.signal import argrelextrema
p = {}
for i in [2,3]:
    p= np.log10(peaklists[i]['Intensity']).copy() #преобразуем интенсивности в десятичные логарифмы
    #Построим функцию плотности
    ints = np.array(p.loc[p>np.log10(0.01)]) #Для ускорения рассчётов - исключим пики с интенсивностями явно шумовыми
    X_plot = np.arange(ints.min()-0.001,ints.max()+0.001,0.00001) #Построим равномерную сетку для построении функции плотности вероятности десятичного логарифма
    Y_kde = FFTKDE(bw ="scott" ).fit(ints)(X_plot) #Определяем плотность вероятности десятичного логарифма интенсивностей с использованием функции построения KDE с применением фурье-преобразования (пакет KDEpy)
    peaks = argrelextrema(Y_kde,np.less) # Находим индексы всех локальных минимумов, для отрисовки соответствующие им значения интенсивностей на графиках
    intensity = 10**X_plot[peaks] # Преобразуем в интенсивности обратно для отрисовки
    plt.figure(figsize=(25,5))
    plt.plot(X_plot,Y_kde)
    plt.scatter(X_plot[peaks],Y_kde[peaks],100,'k','|')
    plt.xlabel("Intensity. log10 scale")
    plt.ylabel("Probabilty density")
    plt.legend(['Intensity KDE function','local minima'])
    plt.xlim((ints.min(),ints.max()))
    # Добавим подписи к минимумам
    for x,y,s in zip(list(X_plot[peaks]),(Y_kde[peaks]),[f'{x:.2f}' for x in intensity]):
        plt.text(x,y+0.05,s,rotation=90,verticalalignment='bottom',horizontalalignment='center')

        

# Выравнивание по пиклисту

In [ ]:
# Настраиваем параметры обработки
from pelmesha.pspectra import Configs
configs = Configs([DataProc_1d,peaks_prop_infunc,msalign,smoothing],
                  smooth_algo = 'GA', 
                  smooth_window = 0.5, 
                  baseline_algo = 'asls', 
                  SNR_threshold = 9, 
                  oversegmentationfilter = 0.05)

# Настройки для функции Getrefpeaks (получение референснного пик-листа с весами относительно которого производить  выравнивание)

step = 50
num_peaks_per_step = 5 
min_occurence = 0.5
return_weight = True

peaklists = {}
# Получаем пик-лист общий
for scan in range(4):
    if scan in [2,3]:
        configs['smooth_algo'] = None
        configs['baseline_algo'] = None
    peaks = ESI_peaks(path, configs, signal_distortions, scan, draw_borders = 2, account_mzscale = account_mzscale)
# Получаем пик-лист референсный
    peaklists[scan] = Getrefpeaks(peaks[scan], 
                                step, 
                                num_peaks_per_step, 
                                min_occurence, 
                                return_weight,
                                CountF = 100,
                                account_mzscale = account_mzscale,
                                draw_borders = 2,
                                return_pkY=True)


In [ ]:
# Создаём итоговый пик-лист после выравнивания и без выравнивания (просто для сравнения)
aln_peaklists = {}
noaln_peaklists = {}
for i in peaklists.keys():
    if i in [2,3]: # Настройки для обработки Орбитреп данных делаем без какой-либо фильтрации пиков, фильтр шумовых пиков сделаем далее. Отстутсвие фильтра на этом этапе - ключевой момент, так как дальнейший фильтр основывается на статистике
        oversegmentationfilter = 0
        SNR_threshold = 0 
        Shift_range = 0.25
        resample_to_dots = None #200000
        CountF = 0
        smooth_algo = None
        smooth_window = 0.5
        baseline_algo = None
    else:
        Shift_range = 0.45
        SNR_threshold = 5
        oversegmentationfilter = 0.1
        resample_to_dots = None
        CountF = 25
        smooth_algo = 'GA'
        smooth_window = 0.5
        baseline_algo = 'asls'
    
    configs = Configs([DataProc_1d,peaks_prop_infunc,msalign,smoothing],
                      align_peaks = peaklists[i][0], 
                      align_pweights = peaklists[i][1],
                      shift_range = Shift_range,
                      resample_to_dots = resample_to_dots,
                  smooth_algo = smooth_algo, 
                  smooth_window = smooth_window, 
                  baseline_algo = baseline_algo, 
                  SNR_threshold = SNR_threshold, 
                  oversegmentationfilter = oversegmentationfilter)
    aln_peaklists.update(ESI_peaks(path, configs,signal_distortions,i, draw_borders = 2.5, account_mzscale = account_mzscale, CountF = CountF, return_pkY = True))
    configs['align_peaks'] = None
    configs['align_pweights'] = None
    noaln_peaklists.update(ESI_peaks(path, configs,signal_distortions,i, draw_borders = 2.5, account_mzscale = account_mzscale, CountF = CountF, return_pkY = True))
    

Предполагаем, что фильтр по интеснивности находится на уровне 1626. Отфильтровываем пики, которые меньше 1626.
Возможно шумовые пики даже выше на самом деле, так как данные с наводкой могут вносить искажения.

In [ ]:
# Фильтруем шумовые пики и сразу же перегруппируем пики.
# account_mzscale = True
bwc = 1
aln_peaklists_gr = {}
noaln_peaklists_gr = {}
for i in [2,3]:
    print(f"aligned data. Scan filter {i}")
    aln_peaklists_gr[i] = Pgrouping_KD(aln_peaklists[i].query("Intensity > 1626"), CountF = 10,bwc = bwc, account_mzscale = account_mzscale,return_pkY=True, sample= "scan filter", roi = f"{i} aligned data")
    print(f"No aligned data. Scan filter {i}")
    noaln_peaklists_gr[i] = Pgrouping_KD(noaln_peaklists[i].query("Intensity > 1626"), CountF = 10, bwc = bwc, account_mzscale = account_mzscale,return_pkY=True, sample= "scan filter", roi = f"{i} no aligned data")


In [ ]:
aln_peaklists[i]

In [ ]:
# Создаём итоговый пик-лист после выравнивания и без выравнивания (просто для сравнения)
aln_peaklists = {}
noaln_peaklists = {}
for i in peaklists.keys():
    if i == 2 or i == 3: # Настройки для обработки Орбитреп данных делаем без какой-либо фильтрации пиков, фильтр шумовых пиков сделаем далее. Отстутсвие фильтра на этом этапе - ключевой момент, так как дальнейший фильтр основывается на статистике
        oversegmentationfilter = 0.05
        SNR_threshold = 5
        # heightfilter = 260000 # фильтр пиков по высоте, выставляем найденное значение
        heightfilter = None
        Shift_range = 0.25
        resample_to_dots = 200000
        CountF = 10
        smooth_algo = None
        smooth_window = 0.5
        baseline_algo = None
        account_mzscale = False
    else:
        Shift_range = 0.45
        SNR_threshold = 3
        oversegmentationfilter = 0.1
        resample_to_dots = None
        heightfilter = None
        CountF = 10
        smooth_algo = 'GA'
        smooth_window = 0.5
        baseline_algo = 'asls'
        account_mzscale = True
    configs = Configs([DataProc_1d,peaks_prop_infunc,msalign,smoothing],
                      align_peaks = peaklists[i][0], 
                      align_pweights = peaklists[i][1],
                      shift_range = Shift_range,
                      resample_to_dots = resample_to_dots,
                  smooth_algo = smooth_algo, 
                  smooth_window = smooth_window, 
                  baseline_algo = baseline_algo, 
                  SNR_threshold = SNR_threshold,
                  heightfilter = heightfilter,
                  oversegmentationfilter = oversegmentationfilter)
    print(configs['SNR_threshold'])
    aln_peaklists.update(ESI_peaks(path, configs,signal_distortions,i, draw_borders = 5, account_mzscale = account_mzscale, CountF = CountF, return_pkY = True))
    configs['align_peaks'] = None
    configs['align_pweights'] = None
    noaln_peaklists.update(ESI_peaks(path, configs,signal_distortions,i, draw_borders = 5, account_mzscale = account_mzscale, CountF = CountF, return_pkY = True))

Отрисовка результата по сканам в определённом диапазоне

In [ ]:
import matplotlib.pyplot as plt

xlim= [1500,1600] # диапазон отрисовки
for reg in aln_peaklists.keys():
    plt.figure(figsize=(20,7.5))
    mz = aln_peaklists[reg]['mz']
    mz_noaln = noaln_peaklists[reg]['mz']
    mz_feat = aln_peaklists_gr[reg]['Peak']

    mz_bool = (mz>xlim[0]) & (mz<xlim[1])
    mz_noaln_bool = (mz_noaln>xlim[0]) & (mz_noaln<xlim[1])
    mz_feat_bool = (mz_feat>xlim[0]) & (mz_feat<xlim[1])

    mz = mz[mz_bool]
    mz_noaln = mz_noaln[mz_noaln_bool]
    mz_feat = mz_feat[mz_feat_bool]

    spectra_ind = aln_peaklists[reg]['spectra_ind'][mz_bool]
    spectra_ind_noaln = noaln_peaklists[reg]['spectra_ind'][mz_noaln_bool]
    spectra_ind_feat = aln_peaklists_gr[reg]['spectra_ind'][mz_feat_bool]

    intensity = aln_peaklists[reg]['Intensity'][mz_bool]
    intensity_noaln = noaln_peaklists[reg]['Intensity'][mz_noaln_bool]
    intensity_feat = intensity

    
    plt.scatter(mz_noaln, spectra_ind_noaln, s = 2, label='Not aligned', color ='g')
    plt.scatter(mz, spectra_ind, s = 4, label='Aligned', alpha=0.75, color='k')
    plt.scatter(mz_feat, spectra_ind_feat, s = 0.75, label='Features', color='r')
    # plt.vlines(mz_feat.unique(), ymin=0, ymax=max(spectra_ind_feat), label='Features', color='r')
    # print(spectra_ind)
    if not spectra_ind.empty:
        plt.ylim(spectra_ind.min(),spectra_ind.max())
    plt.ylabel('Spectra index')
    plt.xlabel('m/z')
    plt.title(f"scan filter {reg}")
    plt.legend(['Aligned', 'Not aligned', 'Features'], markerscale=5)
    plt.xlim(xlim)
    plt.show()


# Вариант выравнивания по усреднённому масс спектру (не доделано)

In [ ]:

mz_min = mz.max() 
mz_max = mz.min()
resample_dots_distance = 0.01

# 3) Если равномерность точек отсутствует, то сделать ресемплинг mz и intens
# 4) Проссумировать интенсивности (разумнее, чем делать индивидуальный пик-пикинг и смотреть по частоте встречаемости: 1) Это LC - если часто встречается - это мусор)
# 5) Построить график
# 6) Найти пики
# 7) Создать референсный список

for i in np.unique(ds['scan_filters']):
    # filtered_scans = scan_dots_slice[np.array(ds['scan_filters'].values == i, dtype=bool)]
    filtered_scans = scan_sequence[np.array(ds['scan_filters'].values == i, dtype=bool)]
    mz_min = mz.max() 
    mz_max = mz.min()
    for scan in filtered_scans:
        mz_min = min(mz_min, *mz[scan_dots_slice[scan]])
        mz_max = max(mz_max, *mz[scan_dots_slice[scan]])
    dots =np.int64( (mz_max - mz_min) / resample_dots_distance)
    print(mz_min, mz_max,dots)

    
    resampled_mz = np.linspace(mz_min, mz_max, dots)
    # Подготовка всех спектров сразу
    all_mz, all_intens = zip(*((mz[scan_dots_slice[scan]], intens[scan_dots_slice[scan]]) for scan in filtered_scans))

    # Векторизованная интерполяция через numpy
    data_int = np.array(tuple(
        np.interp(resampled_mz, mz, intens, 
                left=intens[0], right=intens[-1])
        for mz, intens in zip(all_mz, all_intens)
    ))
    plt.figure(figsize=(20, 5))
    
    plt.plot(resampled_mz, np.sum(data_int, axis=0), label=i)
    plt.xlim(780, 790)

In [ ]:
from pelmesha.pspectra import DataProc_1d, peaks_prop_infunc,msalign,smoothing, Configs
from pelmesha.pfeats import Pgrouping_KD, Getrefpeaks
import pandas as pd


data_int_noaln = {}
data_mz_draw = {}
data_mz_draw_noaln = {}
mz_min = min(mz)
mz_max = max(mz)

resample_to_dots = 150000

for i in np.unique(ds['scan_filters']):
    data_mz_draw[i] = {}
    
    configs = Configs([DataProc_1d,peaks_prop_infunc,msalign,smoothing],
                      align_peaks = peaklists[i][0], 
                      align_pweights = peaklists[i][1], 
                      shift_range = 0.45, 
                      smooth_algo = 'GA', 
                      smooth_window = 0.5,
                      baseline_algo = 'asls', 
                      SNR_threshold = 3, 
                      oversegmentationfilter = 0.05)
    DataProc_configs = configs["DataProc_configs"]
    PeakPicking_configs = configs["peaks_configs"]
    peakl = {}
    filtered_scans = scan_sequence[np.array(ds['scan_filters'].values == i, dtype=bool)]
    data_int = {}
    for n, scan in enumerate(filtered_scans):
        slc = scan_dots_slice[scan]         
        data_mz, data_int[n] = mz[slc], intens[slc]
        if i == 2 or i == 3:
            artefacts_bool = np.zeros(len(data_mz), dtype=bool)
            for distortion in signal_distortions:
                artefacts_bool = artefacts_bool | ((data_mz > distortion[0]) & (data_mz < distortion[1]))
            # artefacts_bool = ((data_mz > 676) & (data_mz < 680)) | ((data_mz > 155) & (data_mz < 157)) | ((data_mz > 168) & (data_mz < 171)) | ((data_mz > 624) & (data_mz < 627)) | ((data_mz > 510) & (data_mz < 512))
            # data_mz, data_int = data_mz[~artefacts_bool], data_int[~artefacts_bool]
            data_int[n][artefacts_bool] = 0
            configs['resample_to_dots'] = resample_to_dots
            old_mz = data_mz
            data_mz = np.linspace(mz_min, mz_max,configs['resample_to_dots'])
            data_mz_draw[i] = data_mz
            data_int[n] = np.interp(np.linspace(mz_min, mz_max,configs['resample_to_dots']), old_mz, data_int[n])
        else:
            data_mz_draw[i][n] = data_mz
        dots_distance = np.median(np.diff(data_mz))
        DataProc_configs['smoothing_configs']['smooth_window'](dots_distance) 
        DataProc_configs['msalign_configs']['shift_range'](dots_distance)
        DataProc_configs['baseliner'](data_mz)

        data_int[n] = DataProc_1d(data_int[n],data_mz,**DataProc_configs)
        peakl[n] = peaks_prop_infunc(data_mz, data_int[n], np.where(np.diff(data_int[n]) != 0)[0], len(data_mz),
                                    scan, **PeakPicking_configs)
    aligned_peaklists[i] = pd.DataFrame(np.vstack(tuple(peakl.values())), columns = PeakPicking_configs['headers'])
    aligned_peaklists[i] = Pgrouping_KD(aligned_peaklists[i], 
                                        CountF= 10, 
                                        # KD_bandwidth='mz_discret',
                                        draw_borders= 3,
                                        return_pkY=True)
    xlim = plt.xlim()
    ax = plt.gca().twinx()
    n = np.random.randint(0, len(filtered_scans))
    slc = scan_dots_slice[filtered_scans[n]]
    mz_plot = mz[slc]
    intens_plot = intens[slc]

    mz_bool = (mz_plot > xlim[0]) & (mz_plot < xlim[1])
    if i == 2 or i == 3:
        mz_draw_bool = (data_mz_draw[i] > xlim[0]) & (data_mz_draw[i] < xlim[1])
        ax.plot(data_mz_draw[i][mz_draw_bool], data_int[n][mz_draw_bool], color = 'green', label = f"Processed mass spectrum N{scan}")
    else:
        mz_draw_bool = (data_mz_draw[i][n] > xlim[0]) & (data_mz_draw[i][n] < xlim[1])
        ax.plot(data_mz_draw[i][n][mz_draw_bool], data_int[n][mz_draw_bool], color = 'green', label = f"Processed mass spectrum N{scan}")
    ax.plot(mz_plot[mz_bool], intens_plot[mz_bool], color = 'red', label = f"Raw mass spectrum N{scan}")
    ax.set_xlabel("m/z")
    ax.set_ylabel("Intensity")
    plt.legend()
    plt.show()

    configs['align_peaks'] = None
    configs['align_pweights'] = None
    
    DataProc_configs = configs["DataProc_configs"]
    PeakPicking_configs = configs["peaks_configs"]
    peakl = {}
    data_mz_draw[i] = {}
    data_int = {}
    for n, scan in enumerate(filtered_scans):
        
        slc = scan_dots_slice[scan]         
        data_mz, data_int[n] = mz[slc], intens[slc]
        if i == 2 or i == 3:
            artefacts_bool = np.zeros(len(data_mz), dtype=bool)
            for distortion in signal_distortions:
                artefacts_bool = artefacts_bool | ((data_mz > distortion[0]) & (data_mz < distortion[1]))
            # artefacts_bool = ((data_mz > 676) & (data_mz < 680)) | ((data_mz > 155) & (data_mz < 157)) | ((data_mz > 168) & (data_mz < 171)) | ((data_mz > 624) & (data_mz < 627)) | ((data_mz > 510) & (data_mz < 512))
            # data_mz, data_int = data_mz[~artefacts_bool], data_int[~artefacts_bool]
            data_int[n][artefacts_bool] = 0
            configs['resample_to_dots'] = resample_to_dots
            old_mz = data_mz
            data_mz = np.linspace(mz_min, mz_max,configs['resample_to_dots'])
            data_mz_draw[i] = data_mz
            data_int[n] = np.interp(np.linspace(mz_min, mz_max,configs['resample_to_dots']), old_mz, data_int[n])
        else:
            data_mz_draw[i][n] = data_mz
        dots_distance = np.median(np.diff(data_mz))
        DataProc_configs['smoothing_configs']['smooth_window'](dots_distance) 
        DataProc_configs['msalign_configs']['shift_range'](dots_distance)
        DataProc_configs['baseliner'](data_mz)

        data_int[n] = DataProc_1d(data_int[n],data_mz,**DataProc_configs)
        peakl[n] = peaks_prop_infunc(data_mz, data_int[n], np.where(np.diff(data_int[n]) != 0)[0], len(data_mz),
                                    scan, **PeakPicking_configs)
    noaln_peaklists[i] = pd.DataFrame(np.vstack(tuple(peakl.values())), columns = PeakPicking_configs['headers'])
    noaln_peaklists[i] = Pgrouping_KD(noaln_peaklists[i], 
                                        CountF= 10, 
                                        # KD_bandwidth='mz_discret',
                                        draw_borders= 3,
                                        return_pkY=True)
    xlim = plt.xlim()
    ax = plt.gca().twinx()


    n = np.random.randint(0, len(filtered_scans))
    slc = scan_dots_slice[filtered_scans[n]]
    mz_plot = mz[slc]
    intens_plot = intens[slc]
    mz_bool = (mz_plot > xlim[0]) & (mz_plot < xlim[1])
    if i == 2 or i == 3:
        mz_draw_bool = (data_mz_draw[i] > xlim[0]) & (data_mz_draw[i] < xlim[1])
        ax.plot(data_mz_draw[i][mz_draw_bool], data_int[n][mz_draw_bool], color = 'green', label = f"Processed mass spectrum N{scan}")
    else:
        mz_draw_bool = (data_mz_draw[i][n] > xlim[0]) & (data_mz_draw[i][n] < xlim[1])
        ax.plot(data_mz_draw[i][n][mz_draw_bool], data_int[n][mz_draw_bool], color = 'green', label = f"Processed mass spectrum N{scan}")
    ax.plot(mz_plot[mz_bool], intens_plot[mz_bool], color = 'red', label = f"Raw mass spectrum N{scan}")
    ax.set_ylabel("Intensity")
    plt.legend()
    plt.show()
